In [17]:
!pip install --quiet pinecone-text google-genai google-auth

In [21]:
import os
import tempfile
import json
from urllib.parse import urlparse
from typing import List, Dict, Any

from google.cloud import aiplatform
from google.cloud import storage
from google import genai
from google.genai import types
from pinecone_text.sparse import BM25Encoder
import google.auth
import google.auth.transport.requests

Some helper functions to deal with object storage.

In [4]:
def parse_gcs_uri(uri):
    """Parses a GCS URI into bucket name and prefix."""
    if not uri.startswith("gs://"):
        raise ValueError("URI must start with gs://")

    parsed = urlparse(uri)
    bucket_name = parsed.netloc
    prefix = parsed.path.lstrip("/")
    return bucket_name, prefix


def download_blob(bucket_name: str, source_blob_name: str, destination_file_name: str):
    """Downloads a blob from the bucket."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(source_blob_name)
    blob.download_to_filename(destination_file_name)
    print(f"Downloaded storage object {source_blob_name} from bucket {bucket_name} to {destination_file_name}.")


Set some initial variables and initialize the client libraries.

In [9]:
project_id = "jkwng-vertex-playground"
location = "us-central1"
index_endpoint_name = "4455874225655250944"
deployed_index_id = "bulbapedia_index"

aiplatform.init(project=project_id, location=location)

client = genai.Client(vertexai=True, project=project_id, location="global")

Here is the query we want to answer using a search over the corpus:

In [6]:
query = "What flying type Pokemon can be found in Kanto region?"

We've pre-computed the BM25 model in GCS and the URI is stored in a Vertex AI artifact called "bm25_index".  We'll retrieve this URI and load it into the BM25 retriever so we can generate the same sparse embeddings used to populate the vector search index.

In [7]:
"""Fetches the URI of the BM25 index artifact from Vertex AI Metadata."""
artifacts = aiplatform.Artifact.list(filter='display_name="bm25_index" AND state="LIVE"')

if not artifacts:
    raise RuntimeError("BM25 index artifact not found.")

bm25_gcs_uri = artifacts[0].uri
print(f"BM25 GCS URI: {bm25_gcs_uri}")

BM25 GCS URI: gs://jkwng-vertex-experiments/pipeline_root/205512073711/bm25-index-generation-pipeline-20251202164421/build-bm25-index_4178641955078537216/bm25_index


Load the encoder.

In [8]:
"""Loads the BM25 encoder from a JSON file in GCS."""
print(f"Loading BM25 encoder from {bm25_gcs_uri}...")
gcs_bucket, gcs_path = parse_gcs_uri(bm25_gcs_uri)

file_path = "bm25_params.json"

with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_file:
    download_blob(gcs_bucket, f"{gcs_path}/{file_path}", temp_file.name)
    temp_file_path = temp_file.name

try:
    bm25_encoder = BM25Encoder()
    bm25_encoder.load(temp_file_path)
    print("BM25 encoder loaded successfully.")
except Exception as e:
    print(f"Error loading BM25 encoder: {e}")
finally:
    if os.path.exists(temp_file_path):
        os.remove(temp_file_path)

Loading BM25 encoder from gs://jkwng-vertex-experiments/pipeline_root/205512073711/bm25-index-generation-pipeline-20251202164421/build-bm25-index_4178641955078537216/bm25_index...
Downloaded storage object pipeline_root/205512073711/bm25-index-generation-pipeline-20251202164421/build-bm25-index_4178641955078537216/bm25_index/bm25_params.json from bucket jkwng-vertex-experiments to /tmp/tmpr5y1kqvh.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


BM25 encoder loaded successfully.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Generate the sparse embeddings for the query text using the BM25 encoder.  This is a wide vector of mostly zeroes, only the indices with values will be returned in two separate arrays.

In [10]:
sparse_vec = bm25_encoder.encode_queries(query)
print(f"Sparse vector: {sparse_vec}")

Sparse vector: {'indices': [991313834, 2045916435, 3962749893, 1810936627, 1029584090, 1547122275], 'values': [0.11906323599409249, 0.04163709175956732, 0.3037160931540277, 0.19494910601309534, 0.16117772067441907, 0.17945675240479816]}


Also generate a dense embeddings using the `gemini-embedding-001` model, which is the same embedding model used in the vector search database.  The vectors in the database are 768 dimension, so we will generate the same length so we can search for similar texts in the database.

In [11]:
model_name = "gemini-embedding-001"

"""Generates a dense embedding using the Google GenAI SDK (Vertex AI backend)."""
result = client.models.embed_content(
    model=model_name,
    contents=[query],
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        title="Query",
        output_dimensionality=768,
    )
)
dense_vec = result.embeddings[0].values
print(f"Dense vector: {dense_vec}")

Dense vector: [-0.031066803261637688, -0.002851264551281929, 0.0045454069040715694, -0.03602393716573715, -0.0035171443596482277, 0.008243756368756294, -0.001401075511239469, -0.00046363973524421453, -0.007822397165000439, -0.01643308810889721, -0.006860668305307627, -0.0012640069471672177, 0.0036180077586323023, 0.02721373736858368, 0.09638076275587082, -0.06183881312608719, -0.009286626242101192, 5.501346095115878e-05, 0.00795945804566145, -0.02555817738175392, 0.01636502705514431, 0.003512181108817458, 0.01166344340890646, -0.025425365194678307, -0.024022497236728668, -0.010825258679687977, -0.0024078760761767626, -0.008431611582636833, 0.027584059163928032, 0.00036923951120115817, -0.0064888461492955685, 0.011048792861402035, 0.016746770590543747, -0.009656153619289398, -0.01778462529182434, 0.009428701363503933, 0.03132135793566704, 0.010195804759860039, -0.016830697655677795, 0.02931852452456951, -0.031708139926195145, 0.005412129685282707, 0.0002576317056082189, 0.02277023717761

Get the public endpoint of the deployed vector index.

In [13]:
index_endpoint = aiplatform.MatchingEngineIndexEndpoint(index_endpoint_name)

# Get the public endpoint domain
public_endpoint_domain = index_endpoint.public_endpoint_domain_name
if not public_endpoint_domain:
    # Fallback or error if no public endpoint
    print("No public endpoint found. Trying private endpoint...")
    # If private endpoint is needed, one might use index_endpoint.private_endpoint_domain_name
    # But for now let's assume public or fail.
    # Note: The SDK might handle this logic, but here we are manual.
    # Let's assume public for this user request context.
    pass

# Construct URL
# https://[endpoint-domain]/v1/projects/[project]/locations/[location]/indexEndpoints/[endpoint-id]/deployedIndexes/[deployed-index-id]:findNeighbors
# Actually the format is: https://[endpoint-domain]/v1/projects/[project]/locations/[location]/indexEndpoints/[endpoint-id]:findNeighbors
# And deployedIndexId is in the body.

# We need the index_endpoint resource ID.
# index_endpoint.resource_name gives projects/.../indexEndpoints/...
# We can parse it or just use the index_endpoint_name if it is the ID.
# The index_endpoint_name passed to the function might be the ID.

api_endpoint = f"https://{public_endpoint_domain}/v1/{index_endpoint.resource_name}:findNeighbors"

print(f"API: {api_endpoint}")

API: https://1952453933.us-central1-205512073711.vdb.vertexai.goog/v1/projects/205512073711/locations/us-central1/indexEndpoints/4455874225655250944:findNeighbors


Now create a hybrid query against our vector search database.  We will send both vectors to the database and the database will find chunks that are most similar to our query.  The answer to the question is likely in one of those chunks.  

Note the parameters here, requesting more nearest neighbors will return more potential answers, but the LLM needs more context (and higher costs/latency) to answer the question.  The goal is to be able to find the right answer among our available data, so we do both dense and sparse search as having two types of retrieval increase our chances of retrieving the right chunk.



In [28]:
neighbor_count = 10
rrf_ranking_alpha = 0.5

In more complex retrieval systems, we could also employ more retrieval systems like structured databases (NL -> SQL), knowledge graphs, or other external tools to pull more relevant context before asking a model to summarize.

To increase accuracy, we can filter away irrelevant chunks using a re-rank step

In [29]:
credentials, _ = google.auth.default()
auth_req = google.auth.transport.requests.Request()
credentials.refresh(auth_req)
token = credentials.token

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

# Construct Payload
payload = {
    "deployedIndexId": deployed_index_id,
    "queries": [
        {
            "datapoint": {
                "featureVector": dense_vec,
                "sparseEmbedding": {
                    "values": sparse_vec['values'],
                    "dimensions": sparse_vec['indices']
                }
            },
            "neighborCount": neighbor_count,
            "rrfRankingAlpha": rrf_ranking_alpha
        }
    ],
    "returnFullDatapoint": True
}

print(json.dumps(payload))

{"deployedIndexId": "bulbapedia_index", "queries": [{"datapoint": {"featureVector": [-0.031066803261637688, -0.002851264551281929, 0.0045454069040715694, -0.03602393716573715, -0.0035171443596482277, 0.008243756368756294, -0.001401075511239469, -0.00046363973524421453, -0.007822397165000439, -0.01643308810889721, -0.006860668305307627, -0.0012640069471672177, 0.0036180077586323023, 0.02721373736858368, 0.09638076275587082, -0.06183881312608719, -0.009286626242101192, 5.501346095115878e-05, 0.00795945804566145, -0.02555817738175392, 0.01636502705514431, 0.003512181108817458, 0.01166344340890646, -0.025425365194678307, -0.024022497236728668, -0.010825258679687977, -0.0024078760761767626, -0.008431611582636833, 0.027584059163928032, 0.00036923951120115817, -0.0064888461492955685, 0.011048792861402035, 0.016746770590543747, -0.009656153619289398, -0.01778462529182434, 0.009428701363503933, 0.03132135793566704, 0.010195804759860039, -0.016830697655677795, 0.02931852452456951, -0.03170813992

Make the call to the search index.  Note we used the REST API here directly instead of the SDK because the embedding metadata is not returned with the SDK. In practice we should store the chunks outside of the vector storage (for example in object storage) - we just stored the actual chunk in the vector database for convenience, but those systems are not designed to hold large unstrctured data.

In [30]:
import requests

response = requests.post(api_endpoint, headers=headers, json=payload)
response.raise_for_status()
response_json = response.json()

# Parse Response
# Response structure: {"nearestNeighbors": [{"id": "query_0", "neighbors": [...]}]}

# TODO: note this is a hacky code for the following reasons:
# - Vertex Vector Search is not designed to hold the actual data, just the vectors - but we stored the chunk in the embedding metadata
# - the embedding_metadata field is not returned by default (we need to pass "returnFullDatapoint")
# - and the SDK MatchNeighbor object does not deserialize the embedding metadata either, only the REST API does
nearest_neighbors = response_json.get("nearestNeighbors", [])
if nearest_neighbors:
    neighbors_list = nearest_neighbors[0].get("neighbors", [])
    print(f"Found {len(neighbors_list)} neighbors.")
    for idx, neighbor in enumerate(neighbors_list):
            datapoint = neighbor.get("datapoint", {})
            neighbor_id = datapoint.get("datapointId")
            distance = neighbor.get("distance")
            sparse_distance = neighbor.get("sparse_distance")
            print(f"Rank {idx+1}: ID={neighbor_id}, Dense Distance={distance}")
        return neighbors_list # Return list of dicts instead of objects
    print("No neighbors found.")

print(neighbors_list)

Found 10 neighbors.
Rank 1: ID=pokemon/bulbapedia/html/gen1/Rhyhorn_(Pok%C3%A9mon).html_chunk_81, Dense Distance=0.7255098819732666
Rank 2: ID=pokemon/bulbapedia/html/gen1/Zapdos_(Pok%C3%A9mon).html_chunk_95, Dense Distance=0.7216657996177673
Rank 3: ID=pokemon/bulbapedia/html/gen1/Magnemite_(Pok%C3%A9mon).html_chunk_92, Dense Distance=0.7106759548187256
Rank 4: ID=pokemon/bulbapedia/html/gen1/Zapdos_(Pok%C3%A9mon).html_chunk_94, Dense Distance=0.7099614143371582
Rank 5: ID=pokemon/bulbapedia/html/gen1/Articuno_(Pok%C3%A9mon).html_chunk_96, Dense Distance=0.7093290686607361
Rank 6: ID=pokemon/bulbapedia/html/gen1/Raticate_(Pok%C3%A9mon).html_chunk_96, Dense Distance=0.694762110710144
Rank 7: ID=pokemon/bulbapedia/html/gen1/Dodrio_(Pok%C3%A9mon).html_chunk_79, Dense Distance=0.688185453414917
Rank 8: ID=pokemon/bulbapedia/html/Bulbasaur_(Pok%C3%A9mon).html_chunk_164, Dense Distance=0.6873714327812195
Rank 9: ID=pokemon/bulbapedia/html/Venusaur_(Pok%C3%A9mon).html_chunk_151, Dense Distan

Now generate the answer to our question with an LLM and the chunks retrieved from the database.

In [32]:
context_chunks = []
blob_cache = {}
storage_client = storage.Client()

for neighbor in neighbors_list:
    datapoint = neighbor.get("datapoint", {})
    metadata = datapoint.get("embeddingMetadata", {})
    
    chunk_uri = metadata.get("chunk_gcs_uri")
    chunk_line_offset = metadata.get("chunk_line_offset")
    
    if chunk_uri and chunk_line_offset is not None:
        if chunk_uri not in blob_cache:
            parsed = urlparse(chunk_uri)
            bucket = storage_client.bucket(parsed.netloc)
            blob = bucket.blob(parsed.path.lstrip("/"))
            content = blob.download_as_text(encoding="utf-8")
            blob_cache[chunk_uri] = content.splitlines()
            
        lines = blob_cache[chunk_uri]
        offset_idx = int(chunk_line_offset)
        if offset_idx < len(lines):
            line = lines[offset_idx]
            chunk_data = json.loads(line)
            chunk_text = chunk_data.get("chunk")
            if chunk_text:
                context_chunks.append(chunk_text)

context_text = "\n".join([f"Context {i+1}: {chunk}" for i, chunk in enumerate(context_chunks)])

prompt = f"""You are a helpful assistant. Answer the user's question using only the provided context.

Context:
{context_text}

Question:
{query}

Answer:
"""

print(f"Prompt:\n{prompt}")



Prompt:
You are a helpful assistant. Answer the user's question using only the provided context.

Context:
Context 1: (Japanese)") | | |  | | --- | | [Safari Zone](/wiki/Kanto_Safari_Zone "Kanto Safari Zone") | | | [Yellow](/wiki/Pok%C3%A9mon_Yellow "Pokémon Yellow") | | |  | | --- | | [Safari Zone](/wiki/Kanto_Safari_Zone "Kanto Safari Zone"), [Cerulean Cave](/wiki/Cerulean_Cave "Cerulean Cave") | | | | |
Context 2: Yellow") | | |  | | --- | | [Power Plant](/wiki/Kanto_Power_Plant "Kanto Power Plant") ([Only one](/wiki/List_of_in-game_event_Pok%C3%A9mon_in_Generation_I#Zapdos "List of in-game event Pokémon in Generation I")) | | | | |
Context 3: (Japanese)") | | |  | | --- | | [Power Plant](/wiki/Kanto_Power_Plant "Kanto Power Plant") | | | [Yellow](/wiki/Pok%C3%A9mon_Yellow "Pokémon Yellow") | | |  | | --- | | [Route 10](/wiki/Kanto_Route_10 "Kanto Route 10"), [Power Plant](/wiki/Kanto_Power_Plant "Kanto Power Plant") | | | | |
Context 4: in Generation I")) | | | [Blue (Japan)](/wiki

In [36]:
model_name = "gemini-2.5-flash"

response = client.models.generate_content(
    model=model_name,
    contents=prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_budget=0,  # Use `0` to turn off thinking
        )
    ),
)

print(response.text)

Zapdos and Moltres are flying type Pokemon that can be found in the Kanto region.
